# 01d Parse Historical Weather Calibration

This notebook parses historical MET Nordic Analysis NetCDF files into calibration-only weather outputs for notebook 04. It is separate from notebook 01a so the canonical realised-weather proxy remains stable. Pressure is used only for calibration and regime support, not as a direct sales-model feature.


## Pipeline position

Notebook 01a defines canonical realised weather, notebook 01b defines deterministic MEPS forecasts, and notebook 01c defines ensemble calibration outputs. This notebook adds optional historical calibration resources for notebook 04. Downstream notebooks should read the processed parquet outputs rather than raw FIMEX folders.


## Inputs and outputs

The notebook reads historical NetCDF folders from the configured FIMEX output directory or a local fallback path: `analysis_temp_nc`, `analysis_precip_nc`, `analysis_wind_nc`, `analysis_humid_nc`, `analysis_cloud_nc`, and `analysis_pressure_nc`. When executed, it writes:

- `data/processed/historical_weather_calibration_windows.parquet`
- `data/processed/historical_weather_calibration_file_audit.parquet`
- `data/processed/historical_weather_calibration_parameters.parquet`
- compact CSV audits under `outputs/historical_weather_calibration/`


## Local-time rule

Raw timestamps are UTC. The output represents the Europe/Oslo local `trade_08_19` window by converting timestamps to local time, retaining hours 08 through 19 inclusive, and aggregating by local `date`, `AvdelingID`, and `aggregation_window`. Partial windows are retained and audited.


## Pressure-regime labels

Pressure is converted from Pa to hPa and used to define simple calibration labels. `transition` means absolute pressure change is at least 4 hPa; `stable_high` means pressure anomaly is at least +5 hPa and absolute change is at most 2 hPa; `low_unsettled` means pressure anomaly is at most -5 hPa; `neutral` covers remaining valid rows; and `unknown` covers missing pressure diagnostics. Tendency is labelled rising, falling, or stable using +/- 1 hPa change thresholds. These labels are audit and calibration constructs, not causal meteorological claims.


In [ ]:

import gc
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import netCDF4
except ImportError as exc:
    raise ImportError("netCDF4 is required for notebook 01d.") from exc


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README_FOR_CODEX.md").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root.")


def resolve_raw_data_root() -> Path:
    candidates = []
    if os.environ.get("FIMEX_OUT_ROOT"):
        candidates.append(Path(os.environ["FIMEX_OUT_ROOT"]))
    candidates += [Path("C:/Users/<USER>/fimex_out"), Path("/mnt/c/Users/<USER>/fimex_out")]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Missing raw Fimex output root.")


PROJECT_ROOT = find_project_root()
RAW_DATA_ROOT = resolve_raw_data_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "historical_weather_calibration"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AVD_LOKASJON = PROJECT_ROOT / "thesis_context" / "Avd_lokasjon.xlsx"
CFG_PATH = RAW_DATA_ROOT / "cloud_points.cfg"
DAILY_OUTPUT_PATH = PROCESSED_DIR / "historical_weather_calibration_windows.parquet"
FILE_AUDIT_OUTPUT_PATH = PROCESSED_DIR / "historical_weather_calibration_file_audit.parquet"
PARAMETER_OUTPUT_PATH = PROCESSED_DIR / "historical_weather_calibration_parameters.parquet"

FILE_COVERAGE_CSV_PATH = OUTPUT_DIR / "01d_file_coverage_by_variable.csv"
DAILY_COVERAGE_CSV_PATH = OUTPUT_DIR / "01d_daily_coverage_by_year_month.csv"
SCHEMA_SUMMARY_CSV_PATH = OUTPUT_DIR / "01d_schema_summary.csv"
VALUE_RANGE_CSV_PATH = OUTPUT_DIR / "01d_value_range_checks.csv"
PRESSURE_REGIME_CSV_PATH = OUTPUT_DIR / "01d_pressure_regime_summary.csv"
PARAMETER_SUMMARY_CSV_PATH = OUTPUT_DIR / "01d_parameter_summary.csv"
VALIDATION_CHECKS_CSV_PATH = OUTPUT_DIR / "01d_validation_checks.csv"

LOCAL_TIMEZONE = "Europe/Oslo"
AGGREGATION_WINDOW = "trade_08_19"
WINDOW_BASIS = "local_time"
EXPECTED_LOCAL_HOURS = list(range(8, 20))
EXPECTED_HOURS = len(EXPECTED_LOCAL_HOURS)
START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2025-07-31")
LAT_LON_TOLERANCE = 0.001
SENTINEL_ABS_THRESHOLD = 1e30
MIN_STORE_MONTH_PRESSURE_OBS = 20
WET_THRESHOLD_MM = 0.1

VARIANTS = {
    "temp": {"folder": "analysis_temp_nc", "ncvar": "air_temperature_2m", "suffix": "temp"},
    "precip": {"folder": "analysis_precip_nc", "ncvar": "precipitation_amount", "suffix": "precip"},
    "wind": {"folder": "analysis_wind_nc", "ncvar": "wind_speed_10m", "suffix": "wind"},
    "humid": {"folder": "analysis_humid_nc", "ncvar": "relative_humidity_2m", "suffix": "humid"},
    "cloud": {"folder": "analysis_cloud_nc", "ncvar": "cloud_area_fraction", "suffix": "cloud"},
    "pressure": {
        "folder": "analysis_pressure_nc",
        "ncvar": "air_pressure_at_sea_level",
        "suffix": "pressure",
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw Fimex root: {RAW_DATA_ROOT}")
print(f"Output folder: {OUTPUT_DIR}")


def parse_cloud_points_cfg(cfg_path: Path) -> tuple[np.ndarray, np.ndarray]:
    text = cfg_path.read_text(encoding="utf-8", errors="replace")
    lat_match = re.search(r"^\s*latitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    lon_match = re.search(r"^\s*longitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    if not lat_match or not lon_match:
        raise ValueError(f"Could not locate latitudeValues/longitudeValues in {cfg_path}")
    lats = np.array([float(x.strip()) for x in lat_match.group(1).split(",") if x.strip()])
    lons = np.array([float(x.strip()) for x in lon_match.group(1).split(",") if x.strip()])
    return lats, lons


def find_col(frame: pd.DataFrame, candidates: list[str]) -> str:
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    raise KeyError(f"None of {candidates} found in columns {list(frame.columns)}")


cfg_lats, cfg_lons = parse_cloud_points_cfg(CFG_PATH)
avd_df = pd.read_excel(AVD_LOKASJON)
col_avd = find_col(avd_df, ["AvdelingID", "Avdeling", "id"])
col_lat = find_col(avd_df, ["Lat", "latitude", "Latitude", "lat"])
col_lon = find_col(avd_df, ["Lon", "longitude", "Longitude", "lon", "Lng"])
if len(avd_df) != len(cfg_lats) or len(cfg_lats) != len(cfg_lons):
    raise ValueError("Store mapping length mismatch.")
lat_diff = np.abs(avd_df[col_lat].to_numpy(dtype="float64") - cfg_lats)
lon_diff = np.abs(avd_df[col_lon].to_numpy(dtype="float64") - cfg_lons)
if int((lat_diff > LAT_LON_TOLERANCE).sum()) or int((lon_diff > LAT_LON_TOLERANCE).sum()):
    raise AssertionError("Store mapping order does not match cloud_points.cfg.")
AVDELING_BY_X = avd_df[col_avd].astype("int64").to_numpy()
N_STORES = len(AVDELING_BY_X)
if N_STORES != 46:
    raise AssertionError(f"Expected 46 stores, found {N_STORES}")
print(f"Stores validated: {N_STORES}")
FILENAME_RE = re.compile(r"^analysis_(\d{8})T(\d{2})Z_(?P<suffix>[A-Za-z0-9_]+)\.nc$")


def timestamp_from_filename(path: Path) -> pd.Timestamp:
    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Unexpected analysis filename: {path.name}")
    ymd = match.group(1)
    hour = int(match.group(2))
    return pd.Timestamp(f"{ymd[:4]}-{ymd[4:6]}-{ymd[6:8]} {hour:02d}:00:00", tz="UTC")


def build_file_index() -> pd.DataFrame:
    rows = []
    for variable, config in VARIANTS.items():
        folder = RAW_DATA_ROOT / config["folder"]
        if not folder.exists():
            rows.append({
                "variable": variable,
                "source_folder": str(folder),
                "path": None,
                "filename": None,
                "file_size_bytes": 0,
                "filename_timestamp_utc": pd.NaT,
                "local_date": pd.NaT,
                "local_hour": np.nan,
                "in_target_window": False,
                "status": "folder_missing",
            })
            continue
        for path in sorted(folder.glob("*.nc")):
            try:
                ts_utc = timestamp_from_filename(path)
                ts_local = ts_utc.tz_convert(LOCAL_TIMEZONE)
                local_date = ts_local.tz_localize(None).normalize()
                local_hour = int(ts_local.hour)
                in_window = bool(START_DATE <= local_date <= END_DATE and local_hour in EXPECTED_LOCAL_HOURS)
                status = "indexed"
            except Exception:
                ts_utc = pd.NaT
                local_date = pd.NaT
                local_hour = np.nan
                in_window = False
                status = "filename_parse_error"
            rows.append({
                "variable": variable,
                "source_folder": str(folder),
                "path": str(path),
                "filename": path.name,
                "file_size_bytes": path.stat().st_size,
                "filename_timestamp_utc": ts_utc,
                "local_date": local_date,
                "local_hour": local_hour,
                "in_target_window": in_window,
                "status": status,
            })
    index = pd.DataFrame(rows)
    print(index.groupby(["variable", "status"], dropna=False).size().reset_index(name="files"))
    print(
        index.groupby("variable", dropna=False)["in_target_window"]
        .sum()
        .reset_index(name="files_in_local_trade_window")
    )
    return index


file_index = build_file_index()


def read_time_utc(ds: netCDF4.Dataset, fallback_ts: pd.Timestamp) -> tuple[pd.Timestamp, str]:
    if "time" not in ds.variables:
        return fallback_ts, "filename_missing_time"
    time_var = ds.variables["time"]
    raw = np.asarray(time_var[:]).ravel()
    if raw.size == 0:
        return fallback_ts, "filename_empty_time"
    units = getattr(time_var, "units", None)
    if not units:
        return fallback_ts, "filename_missing_time_units"
    try:
        decoded = netCDF4.num2date(
            raw[0],
            units=units,
            calendar=getattr(time_var, "calendar", "standard"),
            only_use_cftime_datetimes=False,
        )
        ts = pd.Timestamp(decoded)
        if ts.tzinfo is None:
            ts = ts.tz_localize("UTC")
        else:
            ts = ts.tz_convert("UTC")
        return ts, "netcdf_time"
    except Exception:
        return fallback_ts, "filename_time_decode_error"


def read_vector(ds: netCDF4.Dataset, ncvar: str) -> tuple[np.ndarray, dict]:
    if ncvar not in ds.variables:
        raise KeyError(f"Missing target variable {ncvar}")
    var = ds.variables[ncvar]
    raw = var[:]
    masked = bool(np.ma.isMaskedArray(raw))
    if hasattr(raw, "filled"):
        raw = raw.filled(np.nan)
    arr = np.asarray(raw, dtype="float64").flatten()
    arr = np.where(np.abs(arr) > SENTINEL_ABS_THRESHOLD, np.nan, arr)
    if arr.size != N_STORES:
        raise ValueError(f"Unexpected payload size {arr.size}; expected {N_STORES}")
    return arr, {
        "target_shape": str(tuple(var.shape)),
        "target_dimensions": ",".join(getattr(var, "dimensions", ())),
        "target_dtype": str(var.dtype),
        "target_units": getattr(var, "units", ""),
        "target_standard_name": getattr(var, "standard_name", ""),
        "target_long_name": getattr(var, "long_name", ""),
        "target_cell_methods": getattr(var, "cell_methods", ""),
        "is_masked_array": masked,
    }


def convert_values(variable: str, arr: np.ndarray) -> tuple[np.ndarray, dict]:
    vals = arr.astype("float64", copy=True)
    negative_count = 0
    clipped_count = 0
    if variable == "temp":
        vals = vals - 273.15
        note = "K_to_C"
    elif variable == "precip":
        negative_count = int((vals < 0).sum())
        vals = np.where(vals < 0, np.nan, vals)
        note = "kg_m2_to_mm_equivalent"
    elif variable == "wind":
        negative_count = int((vals < 0).sum())
        vals = np.where(vals < 0, np.nan, vals)
        note = "m_s_nonnegative"
    elif variable in {"humid", "cloud"}:
        finite = vals[np.isfinite(vals)]
        if finite.size and np.nanmax(finite) <= 1.5:
            vals = vals * 100.0
            note = "fraction_to_percent"
        else:
            note = "percent_passthrough"
        clipped_count = int(((vals < 0) | (vals > 100)).sum())
        vals = np.clip(vals, 0.0, 100.0)
    elif variable == "pressure":
        vals = vals / 100.0
        note = "Pa_to_hPa"
    else:
        raise ValueError(variable)
    return vals.astype("float32"), {
        "negative_values_set_missing": negative_count,
        "bounded_values_clipped": clipped_count,
        "conversion_note": note,
    }


def parse_variant(variable: str, index: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    config = VARIANTS[variable]
    target_files = index.loc[(index["variable"] == variable) & (index["in_target_window"])].copy()
    total_capacity = len(target_files) * N_STORES
    dates_arr = np.empty(total_capacity, dtype="datetime64[ns]")
    hours_arr = np.empty(total_capacity, dtype="int8")
    avd_arr = np.empty(total_capacity, dtype="int64")
    values_arr = np.empty(total_capacity, dtype="float32")
    cursor = 0
    audit_rows = []
    t0 = time.time()
    for i, row in enumerate(target_files.itertuples(index=False)):
        path = Path(row.path)
        status = "parsed_ok"
        error_type = None
        error_message = None
        meta = {}
        finite_points = 0
        n_points = 0
        time_source = None
        netcdf_ts = pd.NaT
        try:
            with netCDF4.Dataset(path, "r") as ds:
                netcdf_ts, time_source = read_time_utc(ds, row.filename_timestamp_utc)
                arr, meta = read_vector(ds, config["ncvar"])
                vals, conversion_meta = convert_values(variable, arr)
                meta.update(conversion_meta)
            n_points = int(vals.size)
            finite_points = int(np.isfinite(vals).sum())
            if finite_points == 0:
                status = "all_missing_values"
            else:
                start = cursor
                end = cursor + N_STORES
                dates_arr[start:end] = np.datetime64(row.local_date)
                hours_arr[start:end] = int(row.local_hour)
                avd_arr[start:end] = AVDELING_BY_X
                values_arr[start:end] = vals
                cursor = end
        except Exception as exc:
            status = "parse_error"
            error_type = type(exc).__name__
            error_message = str(exc)[:500]
        audit_rows.append({
            "audit_level": "file_parse",
            "variable": variable,
            "source_folder": str(RAW_DATA_ROOT / config["folder"]),
            "path": str(path),
            "filename": path.name,
            "filename_timestamp_utc": row.filename_timestamp_utc,
            "netcdf_timestamp_utc": netcdf_ts,
            "time_source": time_source,
            "local_date": row.local_date,
            "local_hour": row.local_hour,
            "aggregation_window": AGGREGATION_WINDOW,
            "window_timezone": LOCAL_TIMEZONE,
            "in_target_window": True,
            "status": status,
            "is_available": int(status == "parsed_ok"),
            "is_partial": 0,
            "n_points": n_points,
            "finite_points": finite_points,
            "error_type": error_type,
            "error_message": error_message,
            **meta,
        })
        if (i + 1) % 5000 == 0:
            elapsed = max(time.time() - t0, 1e-6)
            print(
                f"{variable}: parsed {i + 1:,}/{len(target_files):,} "
                f"target-window files ({(i + 1) / elapsed:.0f} files/s)"
            )
    hourly = pd.DataFrame({
        "date": pd.to_datetime(dates_arr[:cursor]),
        "local_hour": hours_arr[:cursor],
        "AvdelingID": avd_arr[:cursor],
        "value": values_arr[:cursor],
    })
    audit = pd.DataFrame(audit_rows)
    print(f"{variable}: hourly rows={len(hourly):,}; file audit rows={len(audit):,}")
    return hourly, audit


def aggregate_variant_hourly(hourly: pd.DataFrame, variable: str) -> pd.DataFrame:
    if hourly.empty:
        return pd.DataFrame(columns=["date", "AvdelingID", f"{variable}_hours_available"])
    hourly = hourly.sort_values(["date", "AvdelingID", "local_hour"])
    grouped = hourly.groupby(["date", "AvdelingID"], observed=True)
    common_hours = {
        f"{variable}_hours_available": ("value", "count"),
        f"{variable}_first_hour_local": ("local_hour", "min"),
        f"{variable}_last_hour_local": ("local_hour", "max"),
    }
    if variable == "temp":
        spec = {
            "temp_hist_mean": ("value", "mean"),
            "temp_hist_min": ("value", "min"),
            "temp_hist_max": ("value", "max"),
            **common_hours,
        }
    elif variable == "precip":
        spec = {"precip_hist_sum": ("value", "sum"), **common_hours}
    elif variable == "wind":
        spec = {"wind_hist_mean": ("value", "mean"), "wind_hist_max": ("value", "max"), **common_hours}
    elif variable == "humid":
        spec = {"humid_hist_mean": ("value", "mean"), **common_hours}
    elif variable == "cloud":
        spec = {"cloud_hist_mean": ("value", "mean"), **common_hours}
    elif variable == "pressure":
        spec = {
            "pressure_hist_mean_hpa": ("value", "mean"),
            "pressure_hist_min_hpa": ("value", "min"),
            "pressure_hist_max_hpa": ("value", "max"),
            "pressure_hist_first_hpa": ("value", "first"),
            "pressure_hist_last_hpa": ("value", "last"),
            **common_hours,
        }
    else:
        raise ValueError(variable)
    out = grouped.agg(**spec).reset_index()
    if variable == "pressure":
        out["pressure_hist_change_hpa_08_19"] = out["pressure_hist_last_hpa"] - out["pressure_hist_first_hpa"]
        out["pressure_hist_abs_change_hpa_08_19"] = out["pressure_hist_change_hpa_08_19"].abs()
        out = out.drop(columns=["pressure_hist_first_hpa", "pressure_hist_last_hpa"])
    return out


variant_daily = {}
variant_audits = []
for variable in VARIANTS:
    hourly, audit = parse_variant(variable, file_index)
    variant_daily[variable] = aggregate_variant_hourly(hourly, variable)
    variant_audits.append(audit)
    del hourly
    gc.collect()
parsed_file_audit = pd.concat(variant_audits, ignore_index=True) if variant_audits else pd.DataFrame()
print("Parsed target-window audit rows:", len(parsed_file_audit))
outside_audit = file_index.loc[~file_index["in_target_window"]].copy()
if not outside_audit.empty:
    outside_audit = outside_audit.assign(
        audit_level="file_index",
        aggregation_window=AGGREGATION_WINDOW,
        window_timezone=LOCAL_TIMEZONE,
        is_available=0,
        is_partial=0,
        n_points=np.nan,
        finite_points=np.nan,
        error_type=pd.NA,
        error_message=pd.NA,
    )
    outside_audit["status"] = np.where(
        outside_audit["status"].eq("indexed"),
        "outside_target_local_window",
        outside_audit["status"],
    )
file_audit = pd.concat([parsed_file_audit, outside_audit], ignore_index=True, sort=False)

full_dates = pd.date_range(START_DATE, END_DATE, freq="D")
grid = pd.MultiIndex.from_product(
    [full_dates, AVDELING_BY_X], names=["date", "AvdelingID"]
).to_frame(index=False)
grid["aggregation_window"] = AGGREGATION_WINDOW
grid["window_timezone"] = LOCAL_TIMEZONE
grid["window_basis"] = WINDOW_BASIS
grid["expected_hours"] = EXPECTED_HOURS

daily = grid.copy()
for variable in VARIANTS:
    daily = daily.merge(variant_daily[variable], on=["date", "AvdelingID"], how="left")

hour_cols = [f"{variable}_hours_available" for variable in VARIANTS]
for col in hour_cols:
    daily[col] = daily[col].fillna(0).astype("int8")
daily["hours_available"] = daily[hour_cols].min(axis=1).astype("int8")
daily["coverage_share"] = (daily["hours_available"].astype("float32") / float(EXPECTED_HOURS)).clip(upper=1.0)
daily["is_full_window"] = (daily["hours_available"] == EXPECTED_HOURS).astype("int8")
daily["month"] = daily["date"].dt.month.astype("int8")

def season_from_month(month: int) -> str:
    if month in (12, 1, 2):
        return "winter"
    if month in (3, 4, 5):
        return "spring"
    if month in (6, 7, 8):
        return "summer"
    return "autumn"

daily["season"] = daily["month"].map(season_from_month).astype("string")
daily["wet_day"] = pd.Series(
    np.where(
        daily["precip_hist_sum"].notna(),
        daily["precip_hist_sum"] >= WET_THRESHOLD_MM,
        pd.NA,
    ),
    dtype="boolean",
)
conditions = [
    daily["cloud_hist_mean"].le(25),
    daily["cloud_hist_mean"].gt(25) & daily["cloud_hist_mean"].lt(75),
    daily["cloud_hist_mean"].ge(75),
]
daily["cloud_regime"] = np.select(conditions, ["open", "partly", "overcast"], default="unknown")

pressure_norm_store_month = (
    daily.loc[daily["pressure_hist_mean_hpa"].notna()]
    .groupby(["AvdelingID", "month"], observed=True)
    .agg(
        pressure_normal_hpa=("pressure_hist_mean_hpa", "mean"),
        pressure_normal_n=("pressure_hist_mean_hpa", "size"),
    )
    .reset_index()
)
pressure_norm_store_month = pressure_norm_store_month.loc[
    pressure_norm_store_month["pressure_normal_n"] >= MIN_STORE_MONTH_PRESSURE_OBS
]
pressure_norm_month = (
    daily.loc[daily["pressure_hist_mean_hpa"].notna()]
    .groupby(["month"], observed=True)
    .agg(month_pressure_normal_hpa=("pressure_hist_mean_hpa", "mean"))
    .reset_index()
)
global_pressure_normal = float(daily["pressure_hist_mean_hpa"].mean())
daily = daily.merge(
    pressure_norm_store_month[["AvdelingID", "month", "pressure_normal_hpa"]],
    on=["AvdelingID", "month"],
    how="left",
)
daily = daily.merge(pressure_norm_month, on="month", how="left")
daily["pressure_normal_fallback_level"] = np.where(
    daily["pressure_normal_hpa"].notna(),
    "store_month",
    np.where(daily["month_pressure_normal_hpa"].notna(), "month", "global"),
)
daily["pressure_normal_hpa"] = (
    daily["pressure_normal_hpa"]
    .fillna(daily["month_pressure_normal_hpa"])
    .fillna(global_pressure_normal)
)
daily = daily.drop(columns=["month_pressure_normal_hpa"])
daily["pressure_anomaly_hpa"] = daily["pressure_hist_mean_hpa"] - daily["pressure_normal_hpa"]
change = daily["pressure_hist_change_hpa_08_19"]
abs_change = daily["pressure_hist_abs_change_hpa_08_19"]
anom = daily["pressure_anomaly_hpa"]
daily["pressure_tendency_regime"] = np.select(
    [change.ge(1.0), change.le(-1.0), change.notna()],
    ["rising", "falling", "stable"],
    default="unknown",
)
daily["pressure_regime"] = np.select(
    [
        abs_change.ge(4.0),
        anom.ge(5.0) & abs_change.le(2.0),
        anom.le(-5.0),
        anom.notna() & abs_change.notna(),
    ],
    ["transition", "stable_high", "low_unsettled", "neutral"],
    default="unknown",
)

float_cols = [
    c
    for c in daily.columns
    if c.endswith(("_mean", "_min", "_max", "_sum", "_hpa", "_share"))
    or c in ["coverage_share", "pressure_anomaly_hpa", "pressure_normal_hpa"]
]
for col in float_cols:
    daily[col] = pd.to_numeric(daily[col], errors="coerce").astype("float32")
for variable in VARIANTS:
    for suffix in ["first_hour_local", "last_hour_local"]:
        col = f"{variable}_{suffix}"
        if col in daily.columns:
            daily[col] = daily[col].fillna(-1).astype("int8")

ordered_cols = [
    "date",
    "AvdelingID",
    "aggregation_window",
    "window_timezone",
    "window_basis",
    "expected_hours",
    "hours_available",
    "coverage_share",
    "is_full_window",
    "month",
    "season",
    "temp_hist_mean",
    "temp_hist_min",
    "temp_hist_max",
    "precip_hist_sum",
    "wind_hist_mean",
    "wind_hist_max",
    "humid_hist_mean",
    "cloud_hist_mean",
    "pressure_hist_mean_hpa",
    "pressure_hist_min_hpa",
    "pressure_hist_max_hpa",
    "pressure_hist_change_hpa_08_19",
    "pressure_hist_abs_change_hpa_08_19",
    "wet_day",
    "cloud_regime",
    "pressure_anomaly_hpa",
    "pressure_regime",
    "pressure_tendency_regime",
    "pressure_normal_hpa",
    "pressure_normal_fallback_level",
    *hour_cols,
]
for variable in VARIANTS:
    ordered_cols.extend([f"{variable}_first_hour_local", f"{variable}_last_hour_local"])
ordered_cols = [c for c in ordered_cols if c in daily.columns]
daily = (
    daily[ordered_cols + [c for c in daily.columns if c not in ordered_cols]]
    .sort_values(["date", "AvdelingID", "aggregation_window"])
    .reset_index(drop=True)
)
print("Daily calibration rows:", len(daily), "columns:", len(daily.columns))
print(daily[["coverage_share", "is_full_window", *hour_cols]].describe().T)
PARAMETER_COLUMNS = [
    "parameter_type",
    "variable",
    "AvdelingID",
    "month",
    "season",
    "aggregation_window",
    "fallback_level",
    "n_obs",
    "n_wet_obs",
    "lower_quantile",
    "upper_quantile",
    "p01",
    "p05",
    "p10",
    "p50",
    "p90",
    "p95",
    "p99",
    "wet_day_probability",
    "conditional_wet_amount_p90",
    "conditional_wet_amount_p95",
    "cloud_open_share",
    "cloud_partly_share",
    "cloud_overcast_share",
    "pressure_regime",
    "pressure_regime_share",
    "pressure_mean_hpa",
    "pressure_std_hpa",
    "pressure_anomaly_p10",
    "pressure_anomaly_p90",
]

def normalize_parameter_rows(rows: list[dict]) -> pd.DataFrame:
    out = pd.DataFrame(rows)
    for col in PARAMETER_COLUMNS:
        if col not in out.columns:
            out[col] = pd.NA
    out = out[PARAMETER_COLUMNS]
    out["AvdelingID"] = pd.to_numeric(out["AvdelingID"], errors="coerce").astype("Int64")
    out["month"] = pd.to_numeric(out["month"], errors="coerce").astype("Int8")
    return out


def group_specs(frame: pd.DataFrame):
    return [
        ("store_month_window", ["AvdelingID", "month", "aggregation_window"]),
        ("month_window", ["month", "aggregation_window"]),
        ("season_window", ["season", "aggregation_window"]),
        ("global_window", ["aggregation_window"]),
    ]


def key_values(name: str, keys, cols: list[str]) -> dict:
    if not isinstance(keys, tuple):
        keys = (keys,)
    row = {
        "fallback_level": name,
        "AvdelingID": pd.NA,
        "month": pd.NA,
        "season": pd.NA,
        "aggregation_window": AGGREGATION_WINDOW,
    }
    for col, val in zip(cols, keys):
        row[col] = val
    return row


def add_continuous_parameters(rows: list[dict], frame: pd.DataFrame, variable: str, value_col: str):
    for fallback, cols in group_specs(frame):
        valid = frame.loc[frame[value_col].notna()]
        for keys, group in valid.groupby(cols, observed=True, dropna=False):
            values = pd.to_numeric(group[value_col], errors="coerce").dropna()
            if values.empty:
                continue
            q = values.quantile([0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99])
            row = key_values(fallback, keys, cols)
            row.update({
                "parameter_type": "continuous_quantiles",
                "variable": variable,
                "n_obs": int(values.size),
                "p01": float(q.loc[0.01]),
                "p05": float(q.loc[0.05]),
                "p10": float(q.loc[0.10]),
                "p50": float(q.loc[0.50]),
                "p90": float(q.loc[0.90]),
                "p95": float(q.loc[0.95]),
                "p99": float(q.loc[0.99]),
                "lower_quantile": float(q.loc[0.01]),
                "upper_quantile": float(q.loc[0.99]),
            })
            rows.append(row)


def add_precip_parameters(rows: list[dict], frame: pd.DataFrame):
    col = "precip_hist_sum"
    for fallback, cols in group_specs(frame):
        valid = frame.loc[frame[col].notna()]
        for keys, group in valid.groupby(cols, observed=True, dropna=False):
            precip = pd.to_numeric(group[col], errors="coerce").dropna()
            if precip.empty:
                continue
            wet = precip >= WET_THRESHOLD_MM
            wet_amount = precip.loc[wet]
            row = key_values(fallback, keys, cols)
            row.update({
                "parameter_type": "precipitation_two_part",
                "variable": "precip",
                "n_obs": int(precip.size),
                "n_wet_obs": int(wet.sum()),
                "wet_day_probability": float(wet.mean()),
                "conditional_wet_amount_p90": (
                    float(wet_amount.quantile(0.90))
                    if not wet_amount.empty else np.nan
                ),
                "conditional_wet_amount_p95": (
                    float(wet_amount.quantile(0.95))
                    if not wet_amount.empty else np.nan
                ),
            })
            rows.append(row)


def add_cloud_parameters(rows: list[dict], frame: pd.DataFrame):
    valid = frame.loc[frame["cloud_regime"].ne("unknown")]
    for fallback, cols in group_specs(valid):
        for keys, group in valid.groupby(cols, observed=True, dropna=False):
            counts = group["cloud_regime"].value_counts(normalize=True)
            row = key_values(fallback, keys, cols)
            row.update({
                "parameter_type": "cloud_regime_shares",
                "variable": "cloud",
                "n_obs": int(len(group)),
                "cloud_open_share": float(counts.get("open", 0.0)),
                "cloud_partly_share": float(counts.get("partly", 0.0)),
                "cloud_overcast_share": float(counts.get("overcast", 0.0)),
            })
            rows.append(row)


def add_pressure_parameters(rows: list[dict], frame: pd.DataFrame):
    valid = frame.loc[frame["pressure_regime"].ne("unknown") & frame["pressure_hist_mean_hpa"].notna()]
    for fallback, cols in group_specs(valid):
        for keys, group in valid.groupby(cols, observed=True, dropna=False):
            base = key_values(fallback, keys, cols)
            pressure = pd.to_numeric(group["pressure_hist_mean_hpa"], errors="coerce").dropna()
            anomaly = pd.to_numeric(group["pressure_anomaly_hpa"], errors="coerce").dropna()
            counts = group["pressure_regime"].value_counts(normalize=True)
            for regime, share in counts.items():
                row = dict(base)
                row.update({
                    "parameter_type": "pressure_regime_shares",
                    "variable": "pressure",
                    "n_obs": int(len(group)),
                    "pressure_regime": regime,
                    "pressure_regime_share": float(share),
                    "pressure_mean_hpa": (
                        float(pressure.mean()) if not pressure.empty else np.nan
                    ),
                    "pressure_std_hpa": (
                        float(pressure.std(ddof=1)) if len(pressure) > 1 else 0.0
                    ),
                    "pressure_anomaly_p10": (
                        float(anomaly.quantile(0.10))
                        if not anomaly.empty else np.nan
                    ),
                    "pressure_anomaly_p90": (
                        float(anomaly.quantile(0.90))
                        if not anomaly.empty else np.nan
                    ),
                })
                rows.append(row)

parameter_rows = []
for variable, value_col in {
    "temp": "temp_hist_mean",
    "wind": "wind_hist_mean",
    "humid": "humid_hist_mean",
    "cloud": "cloud_hist_mean",
    "pressure": "pressure_hist_mean_hpa",
    "pressure_change": "pressure_hist_change_hpa_08_19",
}.items():
    add_continuous_parameters(parameter_rows, daily, variable, value_col)
add_precip_parameters(parameter_rows, daily)
add_cloud_parameters(parameter_rows, daily)
add_pressure_parameters(parameter_rows, daily)
parameters = normalize_parameter_rows(parameter_rows)
print("Parameter rows:", len(parameters), "columns:", len(parameters.columns))
print(parameters.groupby(["parameter_type", "fallback_level"], dropna=False).size().reset_index(name="rows"))
file_coverage = (
    file_audit.groupby(["variable", "status"], dropna=False)
    .agg(files=("filename", "size"), finite_points=("finite_points", "sum"))
    .reset_index()
)
variable_window_coverage = []
for variable in VARIANTS:
    hour_col = f"{variable}_hours_available"
    has_any = daily[hour_col] > 0
    variable_window_coverage.append({
        "variable": variable,
        "daily_rows": int(len(daily)),
        "full_window_rows": int((daily[hour_col] == EXPECTED_HOURS).sum()),
        "full_window_share": float((daily[hour_col] == EXPECTED_HOURS).mean()),
        "mean_hours_available": float(daily[hour_col].mean()),
        "mean_coverage_share": float((daily[hour_col] / EXPECTED_HOURS).mean()),
        "min_date": (
            str(daily.loc[has_any, "date"].min().date())
            if has_any.any() else None
        ),
        "max_date": (
            str(daily.loc[has_any, "date"].max().date())
            if has_any.any() else None
        ),
    })
file_coverage_by_variable = pd.DataFrame(variable_window_coverage).merge(
    file_coverage.pivot_table(
        index="variable",
        columns="status",
        values="files",
        aggfunc="sum",
        fill_value=0,
    ).reset_index(),
    on="variable",
    how="left",
)
daily_coverage_by_year_month = (
    daily.assign(year=daily["date"].dt.year)
    .groupby(["year", "month"], observed=True)
    .agg(
        rows=("date", "size"),
        mean_coverage_share=("coverage_share", "mean"),
        full_window_share=("is_full_window", "mean"),
        temp_mean_hours=("temp_hours_available", "mean"),
        precip_mean_hours=("precip_hours_available", "mean"),
        wind_mean_hours=("wind_hours_available", "mean"),
        humid_mean_hours=("humid_hours_available", "mean"),
        cloud_mean_hours=("cloud_hours_available", "mean"),
        pressure_mean_hours=("pressure_hours_available", "mean"),
    )
    .reset_index()
)
schema_summary = pd.concat(
    [
        pd.DataFrame({
            "output": "daily_windows",
            "column": daily.columns,
            "dtype": [str(t) for t in daily.dtypes],
            "rows": len(daily),
        }),
        pd.DataFrame({
            "output": "file_audit",
            "column": file_audit.columns,
            "dtype": [str(t) for t in file_audit.dtypes],
            "rows": len(file_audit),
        }),
        pd.DataFrame({
            "output": "parameters",
            "column": parameters.columns,
            "dtype": [str(t) for t in parameters.dtypes],
            "rows": len(parameters),
        }),
    ],
    ignore_index=True,
)
value_cols = [
    "temp_hist_mean",
    "temp_hist_min",
    "temp_hist_max",
    "precip_hist_sum",
    "wind_hist_mean",
    "wind_hist_max",
    "humid_hist_mean",
    "cloud_hist_mean",
    "pressure_hist_mean_hpa",
    "pressure_hist_min_hpa",
    "pressure_hist_max_hpa",
    "pressure_hist_change_hpa_08_19",
    "pressure_hist_abs_change_hpa_08_19",
    "pressure_anomaly_hpa",
]
value_range_checks = pd.DataFrame([
    {
        "column": col,
        "non_missing": int(
            pd.to_numeric(daily[col], errors="coerce").notna().sum()
        ),
        "missing": int(
            pd.to_numeric(daily[col], errors="coerce").isna().sum()
        ),
        "min": (
            float(pd.to_numeric(daily[col], errors="coerce").min())
            if pd.to_numeric(daily[col], errors="coerce").notna().any()
            else np.nan
        ),
        "max": (
            float(pd.to_numeric(daily[col], errors="coerce").max())
            if pd.to_numeric(daily[col], errors="coerce").notna().any()
            else np.nan
        ),
        "mean": (
            float(pd.to_numeric(daily[col], errors="coerce").mean())
            if pd.to_numeric(daily[col], errors="coerce").notna().any()
            else np.nan
        ),
    }
    for col in value_cols
])
pressure_regime_summary = (
    daily.groupby(["pressure_regime", "pressure_tendency_regime"], dropna=False)
    .size()
    .reset_index(name="rows")
)
pressure_regime_summary["share"] = pressure_regime_summary["rows"] / len(daily)
parameter_summary = (
    parameters.groupby(["parameter_type", "fallback_level"], dropna=False)
    .agg(
        rows=("variable", "size"),
        variables=("variable", lambda s: ", ".join(sorted(set(map(str, s))))),
    )
    .reset_index()
)

file_coverage_by_variable.to_csv(FILE_COVERAGE_CSV_PATH, index=False)
daily_coverage_by_year_month.to_csv(DAILY_COVERAGE_CSV_PATH, index=False)
schema_summary.to_csv(SCHEMA_SUMMARY_CSV_PATH, index=False)
value_range_checks.to_csv(VALUE_RANGE_CSV_PATH, index=False)
pressure_regime_summary.to_csv(PRESSURE_REGIME_CSV_PATH, index=False)
parameter_summary.to_csv(PARAMETER_SUMMARY_CSV_PATH, index=False)
print("Audit CSVs written under", OUTPUT_DIR)

checks = []
def add_check(name: str, value, passed: bool, note: str = ""):
    checks.append({"check": name, "value": value, "passed": bool(passed), "note": note})
key_cols = ["date", "AvdelingID", "aggregation_window"]
add_check(
    "daily_output_rows",
    len(daily),
    len(daily) == len(pd.date_range(START_DATE, END_DATE, freq="D")) * N_STORES,
)
add_check(
    "daily_duplicate_keys",
    int(daily.duplicated(key_cols).sum()),
    int(daily.duplicated(key_cols).sum()) == 0,
)
add_check(
    "aggregation_window",
    ",".join(sorted(daily["aggregation_window"].unique())),
    set(daily["aggregation_window"].unique()) == {AGGREGATION_WINDOW},
)
add_check(
    "window_timezone",
    ",".join(sorted(daily["window_timezone"].unique())),
    set(daily["window_timezone"].unique()) == {LOCAL_TIMEZONE},
)
add_check(
    "window_basis",
    ",".join(sorted(daily["window_basis"].unique())),
    set(daily["window_basis"].unique()) == {WINDOW_BASIS},
)
add_check("store_count", int(daily["AvdelingID"].nunique()), int(daily["AvdelingID"].nunique()) == N_STORES)
add_check(
    "date_range",
    f"{daily['date'].min().date()} -> {daily['date'].max().date()}",
    daily["date"].min() == START_DATE and daily["date"].max() == END_DATE,
)
first_hour_min = int(np.nanmin(
    daily[[f"{v}_first_hour_local" for v in VARIANTS]]
    .replace(-1, np.nan)
    .to_numpy()
))
last_hour_max = int(np.nanmax(
    daily[[f"{v}_last_hour_local" for v in VARIANTS]]
    .replace(-1, np.nan)
    .to_numpy()
))
add_check(
    "local_hours_used",
    f"{first_hour_min}..{last_hour_max}",
    first_hour_min >= 8 and last_hour_max <= 19,
)
add_check(
    "pressure_hpa_plausible",
    f"{daily['pressure_hist_mean_hpa'].min():.1f}.."
    f"{daily['pressure_hist_mean_hpa'].max():.1f}",
    daily["pressure_hist_mean_hpa"].dropna().between(850, 1100).all(),
)
add_check(
    "humidity_bounds",
    f"{daily['humid_hist_mean'].min():.2f}.."
    f"{daily['humid_hist_mean'].max():.2f}",
    daily["humid_hist_mean"].dropna().between(0, 100).all(),
)
add_check(
    "cloud_bounds",
    f"{daily['cloud_hist_mean'].min():.2f}.."
    f"{daily['cloud_hist_mean'].max():.2f}",
    daily["cloud_hist_mean"].dropna().between(0, 100).all(),
)
add_check(
    "precip_nonnegative",
    float(daily["precip_hist_sum"].min(skipna=True)),
    daily["precip_hist_sum"].dropna().ge(0).all(),
)
add_check(
    "wind_nonnegative",
    float(daily["wind_hist_mean"].min(skipna=True)),
    daily["wind_hist_mean"].dropna().ge(0).all(),
)
add_check("wet_day_exists", "wet_day" in daily.columns, "wet_day" in daily.columns)
add_check(
    "cloud_regime_exists",
    sorted(daily["cloud_regime"].unique().tolist())[:10],
    "cloud_regime" in daily.columns,
)
add_check(
    "pressure_regime_exists",
    sorted(daily["pressure_regime"].unique().tolist()),
    "pressure_regime" in daily.columns,
)
add_check(
    "parameter_fallback_levels",
    ",".join(sorted(parameters["fallback_level"].dropna().unique())),
    parameters["fallback_level"].notna().any(),
)
add_check(
    "parameter_n_obs_present",
    int(parameters["n_obs"].notna().sum()),
    parameters["n_obs"].notna().any(),
)
validation_checks = pd.DataFrame(checks)
validation_checks.to_csv(VALIDATION_CHECKS_CSV_PATH, index=False)
failed = validation_checks.loc[~validation_checks["passed"]]
if not failed.empty:
    print(validation_checks)
    raise AssertionError(f"Validation failed: {failed['check'].tolist()}")

daily.to_parquet(DAILY_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
file_audit.to_parquet(FILE_AUDIT_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
parameters.to_parquet(PARAMETER_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
print("Processed outputs written:")
for path in [DAILY_OUTPUT_PATH, FILE_AUDIT_OUTPUT_PATH, PARAMETER_OUTPUT_PATH]:
    print(f"- {path} ({path.stat().st_size/1_000_000:.2f} MB)")
print(validation_checks)


## Output contract for notebook 04

Notebook 04 should use these outputs only as optional calibration inputs. The daily calibration table can support empirical bounds, wet-day probabilities, conditional wet-precipitation tails, cloud-regime priors, and pressure-regime audit or sensitivity logic. Pressure must remain calibration/regime information and must not be copied into operational forecast rows as target-day realised pressure or used in M2/M4 ML feature lists.
